In [4]:
!pip install -U albumentations

In [5]:
import albumentations as A

print("Albumentations version:", A.__version__)

Albumentations version: 2.0.8


In [6]:
# ============================================================
# 현재 작업 디렉터리 확인
# ============================================================
# Notebook에서 상대 경로를 사용하기 때문에
# 현재 Python의 작업 디렉터리가 어디인지 먼저 확인합니다.
#
# 이후 코드에서는 프로젝트 루트
# pill-object-detection/
# 를 기준으로 상대 경로를 사용합니다.
# ============================================================

from pathlib import Path

print("현재 작업 디렉터리:", Path.cwd())

현재 작업 디렉터리: /Users/apple/dio_folder/python/codeit_cv_project/pill-object-detection/notebooks


In [7]:
# ============================================================
# 프로젝트 루트 디렉터리로 이동
# ============================================================
# 현재 작업 디렉터리가
# .../pill-object-detection/notebooks
# 이므로 한 단계 위인 프로젝트 루트로 이동합니다.
#
# 이후 상대 경로를 아래처럼 통일해서 사용하기 위함입니다.
#
# ./data/...
# ./src/...
# ./outputs/...
# ./config.yaml
# ============================================================

%cd ..

/Users/apple/dio_folder/python/codeit_cv_project/pill-object-detection


In [8]:
from pathlib import Path

print("현재 작업 디렉터리:", Path.cwd())

현재 작업 디렉터리: /Users/apple/dio_folder/python/codeit_cv_project/pill-object-detection


In [9]:
# ============================================================
# 기본 라이브러리 import
# ============================================================
# Faster R-CNN 학습 과정에서 사용할 기본 라이브러리를 불러옵니다.
#
# random / numpy
#   → 실험 재현성을 위한 random seed 설정
#
# sys / pathlib
#   → 프로젝트 내부 Python 모듈 및 경로 관리
#
# defaultdict
#   → combination_key 기준 데이터 그룹화
#
# torch
#   → PyTorch 모델 학습
#
# train_test_split
#   → train / validation / test 데이터 분할
#
# DataLoader / Subset
#   → PyTorch Dataset을 학습용 batch로 구성
# ============================================================

import random
import sys

from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

In [10]:
# ============================================================
# Dataset / Transform 모듈 import
# ============================================================

# 현재 프로젝트 구조:
#
# pill-object-detection/
# └── src/
#     ├── PillDetectionDataset.py
#     └── pill_transforms.py
#
# src 폴더를 Python 모듈 검색 경로에 추가한 뒤,
# Dataset 클래스와 baseline transform 함수를 import합니다.
# ============================================================

SRC_DIR = Path("./src").resolve()

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from PillDetectionDataset import (
    PillDetectionDataset,
    detection_collate_fn,
)

from pill_transforms import get_valid_transforms

print("src 경로:", SRC_DIR)
print("PillDetectionDataset import 완료")
print("pill_transforms import 완료")

src 경로: /Users/apple/dio_folder/python/codeit_cv_project/pill-object-detection/src
PillDetectionDataset import 완료
pill_transforms import 완료


In [11]:
# ============================================================
# config.yaml 불러오기
# ============================================================
# 프로젝트 루트의 config.yaml에서
# 학습에 필요한 설정값을 관리합니다.
#
# 예:
# - random seed
# - batch size
# - learning rate
# - epoch
# - optimizer
# - scheduler
# - checkpoint 경로
# - W&B 설정
#
# Notebook에서는 @hydra.main보다
# OmegaConf.load() 방식이 실행 흐름을 확인하기 쉽습니다.
# ============================================================

from omegaconf import OmegaConf


# 프로젝트 루트 기준 config.yaml 경로
config_path = Path("./config.yaml")


# config 파일 존재 여부 확인
assert config_path.exists(), (
    f"config.yaml 파일이 없습니다: "
    f"{config_path.resolve()}"
)


# YAML 설정 불러오기
cfg = OmegaConf.load(
    config_path
)


# 현재 설정 내용 출력
print(
    OmegaConf.to_yaml(cfg)
)

project:
  name: pill-object-detection
  seed: 42
paths:
  project_root: .
  dataset_root: ${paths.project_root}/data/dataset/cleaning_data/sprint_ai_project1_data_260809_baseline_dataset
  checkpoint_dir: ${paths.project_root}/outputs/checkpoints
  prediction_dir: ${paths.project_root}/outputs/predictions
  submission_dir: ${paths.project_root}/outputs/submissions
dataset:
  image_dir_name: train_images
  annotation_dir_name: train_annotations
  label_offset: 1
  strict: false
  validate_image_size: true
  train_ratio: 0.8
  val_ratio: 0.1
  test_ratio: 0.1
dataloader:
  batch_size: 4
  num_workers: 0
  pin_memory: true
model:
  name: fasterrcnn_resnet50_fpn
  pretrained: true
  trainable_backbone_layers: 3
  min_size: 640
  max_size: 640
train:
  epochs: 20
  optimizer: sgd
  learning_rate: 0.005
  momentum: 0.9
  weight_decay: 0.0005
  scheduler:
    type: step
    step_size: 5
    gamma: 0.1
  gradient_clip: null
  save_best: true
wandb:
  enabled: true
  project: pill-object-detec

In [12]:
# ============================================================
# Dataset 경로 설정 및 폴더 존재 여부 확인
# ============================================================
# 현재 데이터 구조:
#
# pill-object-detection/
# └── data/
#     └── dataset/
#         └── cleaning_data/
#             └── sprint_ai_project1_data_260809_baseline_dataset/
#                   ├── train_images/
#                   ├── train_annotations/
#                   └── test_images/
#
# PillDetectionDataset의 root에는
# train_images 자체가 아니라
# train_images와 train_annotations의 상위 폴더인
# cleaning_data를 전달합니다.
# ============================================================

dataset_root = Path(
    "./data/dataset/cleaning_data/"
    "sprint_ai_project1_data_260809_baseline_dataset"
)

train_image_dir = (
    dataset_root / "train_images"
)

train_annotation_dir = (
    dataset_root / "train_annotations"
)

test_image_dir = (
    dataset_root / "test_images"
)

print(
    "Dataset root:",
    dataset_root.resolve(),
)

print(
    "train_images 존재:",
    train_image_dir.exists(),
)

print(
    "train_annotations 존재:",
    train_annotation_dir.exists(),
)

print(
    "test_images 존재:",
    test_image_dir.exists(),
)

# 필수 폴더가 없으면 여기서 바로 중단
assert dataset_root.exists(), (
    f"Dataset root가 없습니다: "
    f"{dataset_root.resolve()}"
)

assert train_image_dir.exists(), (
    f"train_images 폴더가 없습니다: "
    f"{train_image_dir.resolve()}"
)

assert train_annotation_dir.exists(), (
    f"train_annotations 폴더가 없습니다: "
    f"{train_annotation_dir.resolve()}"
)

assert test_image_dir.exists(), (
    f"test_images 폴더가 없습니다: "
    f"{test_image_dir.resolve()}"
)

print("Dataset 경로 확인 완료")

Dataset root: /Users/apple/dio_folder/python/codeit_cv_project/pill-object-detection/data/dataset/cleaning_data/sprint_ai_project1_data_260809_baseline_dataset
train_images 존재: True
train_annotations 존재: True
test_images 존재: True
Dataset 경로 확인 완료


In [13]:
# ============================================================
# PillDetectionDataset 생성 및 기본 동작 확인
# ============================================================
# 민협님이 작성한 PillDetectionDataset.py를 그대로 사용합니다.
#
# transforms=None
#   → 현재는 transform을 적용하지 않습니다.
#   → 이미지 타입은 PIL.Image.Image로 반환됩니다.
#
# label_offset=1
#   → Faster R-CNN은 0번 클래스를 background로 사용하므로
#     실제 알약 클래스는 1부터 시작합니다.
#
# strict=False
#   → 일부 annotation 누락/불일치가 있어도
#     예외로 중단하지 않고 경고 후 가능한 샘플을 사용합니다.
#
# validate_image_size=True
#   → JSON의 width/height와 실제 이미지 크기가 일치하는지 확인합니다.
# ============================================================

dataset = PillDetectionDataset(
    root=dataset_root,
    transforms=None,
    label_offset=1,
    strict=False,
    validate_image_size=True,
)


# ============================================================
# Dataset 기본 정보 출력
# ============================================================

print(
    f"Number of images: "
    f"{len(dataset)}"
)

print(
    f"Number of pill classes: "
    f"{dataset.num_classes}"
)


# ============================================================
# 첫 번째 샘플 요약 확인
# ============================================================
# get_sample_summary()는 이미지를 실제로 읽지 않고
# 해당 샘플의 기본 annotation 정보를 확인하는 함수입니다.
# ============================================================

print("\n===== First sample summary =====")

print(
    dataset.get_sample_summary(0)
)

Number of images: 217
Number of pill classes: 56

===== First sample summary =====
{'index': 0, 'file_name': 'K-001900-016548-019607-029451_0_2_0_2_70_000_200.png', 'pill_ids': ['001900', '016548', '019607', '029451'], 'num_pills': 4, 'labels': [1, 16, 23, 43], 'category_ids': [1900, 16548, 19607, 29451], 'camera_angle': 70}


/Users/apple/dio_folder/python/codeit_cv_project/pill-object-detection/src/PillDetectionDataset.py:268: RuntimeWarning: 이미지 K-003351-020014-020238_0_2_0_2_75_000_200.png의 알약 ID 수는 3개지만 대응 JSON은 0개입니다.
  self._handle_problem(


In [14]:
# ============================================================
# Dataset 첫 번째 샘플 상세 확인
# ============================================================
# PillDetectionDataset의 __getitem__이 실제로 반환하는
#
# 1. image
# 2. target
# 3. metadata
#
# 구조를 확인합니다.
#
# 특히 Faster R-CNN 학습에 필요한
#
# boxes
# labels
# image_id
# area
# iscrowd
#
# 가 올바르게 생성되었는지 확인합니다.
# ============================================================

image, target, metadata = dataset[0]


# ============================================================
# Image 확인
# ============================================================
# 아직 transform을 적용하지 않았으므로
# PIL.Image.Image가 반환되는 것이 정상입니다.
# ============================================================

print("===== Image =====")

print(
    "Image type:",
    type(image),
)

print(
    "Image size:",
    image.size,
)


# ============================================================
# Target 확인
# ============================================================
# boxes는 Faster R-CNN이 사용하는
# [x1, y1, x2, y2] 형식이어야 합니다.
#
# labels는 torch.int64 타입이며
# 0은 background이므로 실제 클래스는 1부터 시작합니다.
# ============================================================

print("\n===== Target =====")

print(
    "Target keys:",
    target.keys(),
)

print(
    "Boxes shape:",
    target["boxes"].shape,
)

print(
    "Boxes:",
    target["boxes"],
)

print(
    "Labels:",
    target["labels"],
)

print(
    "Image ID:",
    target["image_id"],
)

print(
    "Area:",
    target["area"],
)

print(
    "iscrowd:",
    target["iscrowd"],
)


# ============================================================
# Metadata 확인
# ============================================================
# metadata는 Faster R-CNN 자체 학습에는 직접 사용하지 않지만
# 추후 예측 결과 분석이나 원본 알약 ID 추적 등에 사용할 수 있습니다.
# ============================================================

print("\n===== Metadata =====")

print(
    "File name:",
    metadata["file_name"],
)

print(
    "Combination key:",
    metadata["combination_key"],
)

print(
    "Pill IDs:",
    metadata["pill_ids"],
)

print(
    "Number of pills:",
    metadata["num_pills"],
)

===== Image =====
Image type: <class 'PIL.Image.Image'>
Image size: (976, 1280)

===== Target =====
Target keys: dict_keys(['boxes', 'labels', 'image_id', 'area', 'iscrowd', 'annotation_id', 'ignore'])
Boxes shape: torch.Size([4, 4])
Boxes: tensor([[ 644.,  845.,  833., 1035.],
        [ 144.,  799.,  383., 1038.],
        [ 657.,  287.,  812.,  437.],
        [ 108.,  218.,  500.,  482.]])
Labels: tensor([ 1, 16, 23, 43])
Image ID: tensor([0])
Area: tensor([ 35910.,  57121.,  23250., 103488.])
iscrowd: tensor([0, 0, 0, 0])

===== Metadata =====
File name: K-001900-016548-019607-029451_0_2_0_2_70_000_200.png
Combination key: K-001900-016548-019607-029451
Pill IDs: ['001900', '016548', '019607', '029451']
Number of pills: 4


In [15]:
# ============================================================
# combination_key 기준 이미지 그룹화
# ============================================================
# 동일한 알약 조합으로 촬영된 이미지가
# Train / Validation / Test에 동시에 포함되는 것을 방지하기 위해
# 이미지 단위가 아니라 combination_key 단위로 그룹화합니다.
#
# 예:
# K-001900-016548-019607-029451
#
# 동일한 combination_key를 가진 모든 이미지는
# 하나의 split에만 들어가도록 합니다.
# ============================================================

group_to_indices = defaultdict(list)


for index, sample in enumerate(
    dataset.samples
):

    combination_key = (
        sample["metadata"]
        ["combination_key"]
    )

    group_to_indices[
        combination_key
    ].append(index)


# 정렬하여 split 재현성을 높입니다.
group_keys = sorted(
    group_to_indices.keys()
)


# ============================================================
# 그룹 정보 확인
# ============================================================

print(
    f"전체 이미지 수: "
    f"{len(dataset)}"
)

print(
    f"전체 combination_key 수: "
    f"{len(group_keys)}"
)

print(
    "\n첫 번째 combination_key:"
)

print(
    group_keys[0]
)

print(
    "해당 그룹의 이미지 index:",
    group_to_indices[
        group_keys[0]
    ],
)

전체 이미지 수: 217
전체 combination_key 수: 113

첫 번째 combination_key:
K-001900-016548-019607-029451
해당 그룹의 이미지 index: [0, 1, 2]


In [16]:
# ============================================================
# combination_key 기준 Train / Validation / Test 분할
# ============================================================
# 동일한 combination_key가 여러 split에 섞이지 않도록
# 이미지가 아니라 group 단위로 분할합니다.
#
# 분할 비율과 random seed는 config.yaml 값을 사용합니다.
# ============================================================


# ============================================================
# 1. split 비율 검증
# ============================================================
# Train + Validation + Test 비율의 합이 1.0인지 확인합니다.
# ============================================================

assert abs(
    cfg.dataset.train_ratio
    + cfg.dataset.val_ratio
    + cfg.dataset.test_ratio
    - 1.0
) < 1e-8, (
    "train_ratio + val_ratio + test_ratio의 합은 "
    "1.0이어야 합니다."
)


# ============================================================
# 2. 1차 분할
# ============================================================
# 전체 group에서 Train을 제외한
# Validation + Test 비율만큼 temp_groups로 분리합니다.
# ============================================================

temp_ratio = (
    cfg.dataset.val_ratio
    + cfg.dataset.test_ratio
)

train_groups, temp_groups = train_test_split(
    group_keys,
    test_size=temp_ratio,
    random_state=cfg.project.seed,
    shuffle=True,
)


# ============================================================
# 3. 2차 분할
# ============================================================
# temp_groups 내부에서 Validation / Test를 다시 분리합니다.
# ============================================================

relative_test_ratio = (
    cfg.dataset.test_ratio
    / temp_ratio
)

valid_groups, test_groups = train_test_split(
    temp_groups,
    test_size=relative_test_ratio,
    random_state=cfg.project.seed,
    shuffle=True,
)


# ============================================================
# 4. 중복 검증을 쉽게 하기 위해 set으로 변환
# ============================================================

train_groups = set(train_groups)
valid_groups = set(valid_groups)
test_groups = set(test_groups)


# ============================================================
# 5. 분할된 combination_key 개수 확인
# ============================================================

print(
    "Train combination_key 수:",
    len(train_groups),
)

print(
    "Validation combination_key 수:",
    len(valid_groups),
)

print(
    "Test combination_key 수:",
    len(test_groups),
)

print(
    "전체 combination_key 수:",
    (
        len(train_groups)
        + len(valid_groups)
        + len(test_groups)
    ),
)

Train combination_key 수: 90
Validation combination_key 수: 11
Test combination_key 수: 12
전체 combination_key 수: 113


In [17]:
# ============================================================
# combination_key 그룹을 Dataset index로 변환
# ============================================================
# 현재 train_groups / valid_groups / test_groups에는
# combination_key 문자열만 들어 있습니다.
#
# PyTorch의 Subset을 만들기 위해서는
# 실제 Dataset의 index 목록이 필요하므로,
# 각 combination_key에 속한 이미지 index를 모아서 변환합니다.
# ============================================================


def groups_to_indices(
    groups,
    group_mapping,
):
    # 각 combination_key에 속한 이미지 index를 하나의 리스트로 합친 뒤
    # 항상 같은 순서를 유지하도록 정렬해서 반환합니다.
    return sorted(
        index
        for group in groups
        for index in group_mapping[group]
    )


# Train 이미지 index
train_indices = groups_to_indices(
    train_groups,
    group_to_indices,
)

# Validation 이미지 index
valid_indices = groups_to_indices(
    valid_groups,
    group_to_indices,
)

# Test 이미지 index
test_indices = groups_to_indices(
    test_groups,
    group_to_indices,
)


# ============================================================
# 실제 이미지 개수 확인
# ============================================================
# combination_key마다 이미지 수가 서로 다를 수 있기 때문에
# 그룹 비율 8:1:1과 이미지 비율이 정확히 8:1:1이 되지는 않을 수 있습니다.
# ============================================================

print(
    "Train 이미지 수:",
    len(train_indices),
)

print(
    "Validation 이미지 수:",
    len(valid_indices),
)

print(
    "Test 이미지 수:",
    len(test_indices),
)

print(
    "전체 이미지 수:",
    (
        len(train_indices)
        + len(valid_indices)
        + len(test_indices)
    ),
)

Train 이미지 수: 176
Validation 이미지 수: 16
Test 이미지 수: 25
전체 이미지 수: 217


In [18]:
# ============================================================
# Train / Validation / Test split 검증
# ============================================================
# 현재 combination_key 기준으로 데이터를 나눴지만,
# 실제로 세 split 사이에 중복된 그룹이나 이미지 index가 없는지
# 한 번 더 검증합니다.
#
# 검증 항목:
#
# 1. Train / Validation combination_key 중복 없음
# 2. Train / Test combination_key 중복 없음
# 3. Validation / Test combination_key 중복 없음
# 4. 모든 이미지가 정확히 한 번씩만 분배됨
# ============================================================


# ============================================================
# 1. combination_key 중복 검증
# ============================================================

assert train_groups.isdisjoint(
    valid_groups
), (
    "Train과 Validation에 "
    "중복 combination_key가 있습니다."
)

assert train_groups.isdisjoint(
    test_groups
), (
    "Train과 Test에 "
    "중복 combination_key가 있습니다."
)

assert valid_groups.isdisjoint(
    test_groups
), (
    "Validation과 Test에 "
    "중복 combination_key가 있습니다."
)


# ============================================================
# 2. 이미지 index 중복 검증
# ============================================================

train_index_set = set(
    train_indices
)

valid_index_set = set(
    valid_indices
)

test_index_set = set(
    test_indices
)


assert train_index_set.isdisjoint(
    valid_index_set
), (
    "Train과 Validation에 "
    "중복 이미지 index가 있습니다."
)

assert train_index_set.isdisjoint(
    test_index_set
), (
    "Train과 Test에 "
    "중복 이미지 index가 있습니다."
)

assert valid_index_set.isdisjoint(
    test_index_set
), (
    "Validation과 Test에 "
    "중복 이미지 index가 있습니다."
)


# ============================================================
# 3. 전체 이미지가 정확히 한 번씩 분배됐는지 확인
# ============================================================

all_split_indices = (
    train_indices
    + valid_indices
    + test_indices
)


assert len(
    all_split_indices
) == len(
    dataset
), (
    "분할된 이미지 개수와 "
    "전체 Dataset 이미지 개수가 일치하지 않습니다."
)


assert len(
    set(all_split_indices)
) == len(
    dataset
), (
    "하나의 이미지가 두 개 이상의 split에 "
    "중복 포함되어 있습니다."
)


# ============================================================
# 검증 결과 출력
# ============================================================

print("===== Split 검증 완료 =====")

print(
    "Train      :",
    len(train_indices),
    "장",
)

print(
    "Validation :",
    len(valid_indices),
    "장",
)

print(
    "Test       :",
    len(test_indices),
    "장",
)

print(
    "전체       :",
    len(all_split_indices),
    "장",
)

print(
    "\ncombination_key 중복 없음"
)

print(
    "이미지 index 중복 없음"
)

===== Split 검증 완료 =====
Train      : 176 장
Validation : 16 장
Test       : 25 장
전체       : 217 장

combination_key 중복 없음
이미지 index 중복 없음


In [19]:
# ============================================================
# Faster R-CNN Baseline Transform 설정
# ============================================================

# 이번 baseline에서는 augmentation을 사용하지 않습니다.
#
# 팀원이 작성한 pill_transforms.py의
# get_valid_transforms()를 Train / Validation 모두 사용합니다.
#
# image_size=640
# → YOLO baseline의 imgsz=640과 입력 크기 기준 통일
#
# to_tensor=False
# → pill_transforms.py 내부의 Normalize + ToTensorV2는 사용하지 않음
# → Faster R-CNN 입력에 맞는 Tensor 변환은 아래 Adapter에서 수행
#
# 최종 이미지:
# torch.Tensor
# [C, H, W]
# float32
# [0, 1]
# ============================================================


class FasterRCNNTransform:
    """
    Albumentations 기본 transform 결과를
    Faster R-CNN 입력 형식으로 변환하는 Adapter.
    """

    def __init__(self, transform):
        self.transform = transform

    def __call__(
        self,
        image,
        bboxes,
        labels,
    ):
        transformed = self.transform(
            image=image,
            bboxes=bboxes,
            labels=labels,
        )

        image = transformed["image"]

        # HWC numpy.ndarray
        # → CHW torch.Tensor
        # → float32 [0, 1]
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(
                np.ascontiguousarray(image)
            )

            if image.ndim == 3:
                image = image.permute(
                    2,
                    0,
                    1,
                )

            image = image.float() / 255.0

        transformed["image"] = image

        return transformed


# ============================================================
# Baseline Transform 생성
# ============================================================

IMAGE_SIZE = 640

base_transform = get_valid_transforms(
    image_size=IMAGE_SIZE,
    to_tensor=False,
)

# Baseline에서는 Train / Validation 모두
# augmentation 없는 동일한 기본 transform 사용
train_transforms = FasterRCNNTransform(
    base_transform
)

eval_transforms = FasterRCNNTransform(
    base_transform
)


# ============================================================
# Transform 설정 확인
# ============================================================

print(
    "Train transform:",
    train_transforms.transform,
)

print(
    "\nEval transform:",
    eval_transforms.transform,
)

Train transform: Compose([
  LongestMaxSize(p=1.0, area_for_downscale=None, interpolation=1, mask_interpolation=0, max_size=640, max_size_hw=None),
  PadIfNeeded(p=1.0, border_mode=0, fill=0.0, fill_mask=0.0, min_height=640, min_width=640, pad_height_divisor=None, pad_width_divisor=None, padding=0, position='center'),
], p=1.0, bbox_params={'format': 'pascal_voc', 'label_fields': ['labels'], 'min_area': 0.0, 'min_visibility': 0.0, 'min_width': 0.0, 'min_height': 0.0, 'check_each_transform': True, 'clip': True, 'max_accept_ratio': None}, keypoint_params=None, additional_targets={}, is_check_shapes=True)

Eval transform: Compose([
  LongestMaxSize(p=1.0, area_for_downscale=None, interpolation=1, mask_interpolation=0, max_size=640, max_size_hw=None),
  PadIfNeeded(p=1.0, border_mode=0, fill=0.0, fill_mask=0.0, min_height=640, min_width=640, pad_height_divisor=None, pad_width_divisor=None, padding=0, position='center'),
], p=1.0, bbox_params={'format': 'pascal_voc', 'label_fields': ['label

In [20]:
# ============================================================
# Train용 / Evaluation용 Dataset 생성
# ============================================================

# 같은 원본 데이터와 같은 sample 순서를 사용하되,
#
# Train Dataset
# → baseline train_transforms 적용
#
# Validation / Test Dataset
# → baseline eval_transforms 적용
#
# 이번 baseline에서는 augmentation을 사용하지 않으며,
# train / val / test 모두 640 기준 기본 transform을 사용합니다.
# ============================================================

# ============================================================
# Train Dataset
# ============================================================

train_base_dataset = PillDetectionDataset(
    root=dataset_root,
    transforms=train_transforms,
    label_offset=1,
    strict=False,
    validate_image_size=True,
)

# ============================================================
# Validation / Test Dataset
# ============================================================

eval_base_dataset = PillDetectionDataset(
    root=dataset_root,
    transforms=eval_transforms,
    label_offset=1,
    strict=False,
    validate_image_size=True,
)

# ============================================================
# Dataset 크기 확인
# ============================================================

print(
    "Split 기준 Dataset:",
    len(dataset),
)

print(
    "Train base Dataset:",
    len(train_base_dataset),
)

print(
    "Eval base Dataset:",
    len(eval_base_dataset),
)

Split 기준 Dataset: 217
Train base Dataset: 217
Eval base Dataset: 217


In [21]:
# ============================================================
# Split 기준 / Train / Eval Dataset의 sample 순서 검증
# ============================================================
# 앞에서 생성한 train_indices / valid_indices / test_indices는
# dataset.samples의 순서를 기준으로 만들어졌습니다.
#
# 따라서 train_base_dataset과 eval_base_dataset의
# sample 순서가 dataset과 동일해야
# 같은 index를 안전하게 사용할 수 있습니다.
# ============================================================

assert len(dataset) == len(train_base_dataset)
assert len(dataset) == len(eval_base_dataset)


for index in range(len(dataset)):

    split_file = (
        dataset.samples[index]
        ["metadata"]
        ["file_name"]
    )

    train_file = (
        train_base_dataset.samples[index]
        ["metadata"]
        ["file_name"]
    )

    eval_file = (
        eval_base_dataset.samples[index]
        ["metadata"]
        ["file_name"]
    )

    assert (
        split_file
        == train_file
        == eval_file
    ), (
        f"Dataset sample 순서가 다릅니다. "
        f"index={index}, "
        f"split={split_file}, "
        f"train={train_file}, "
        f"eval={eval_file}"
    )


print("Dataset sample 순서 검증 완료")

Dataset sample 순서 검증 완료


In [22]:
# ============================================================
# Train / Validation / Test Subset 생성
# ============================================================
# 앞에서 만든 train_indices / valid_indices / test_indices를
# 각각 Train용 Dataset과 Eval용 Dataset에 적용합니다.
#
# Train
# → train_base_dataset 사용
# → augmentation 없는 baseline train transform 적용
#
# Validation / Test
# → eval_base_dataset 사용
# → augmentation 없는 baseline eval transform 적용
# ============================================================


train_dataset = Subset(
    train_base_dataset,
    train_indices,
)


valid_dataset = Subset(
    eval_base_dataset,
    valid_indices,
)


test_dataset = Subset(
    eval_base_dataset,
    test_indices,
)


# ============================================================
# Subset 크기 확인
# ============================================================

print(
    "Train Dataset:",
    len(train_dataset),
)

print(
    "Validation Dataset:",
    len(valid_dataset),
)

print(
    "Test Dataset:",
    len(test_dataset),
)

Train Dataset: 176
Validation Dataset: 16
Test Dataset: 25


In [23]:
# ============================================================
# Train / Validation / Test DataLoader 생성
# ============================================================

# Object Detection에서는 이미지마다 객체 수가 다르기 때문에
# detection_collate_fn을 사용합니다.
#
# Train
# → shuffle=True
#
# Validation / Test
# → shuffle=False
#
# DataLoader 설정값은 config.yaml에서 불러옵니다.
# ============================================================

batch_size = cfg.dataloader.batch_size
num_workers = cfg.dataloader.num_workers

# CUDA 환경에서만 pin_memory 활성화
pin_memory = (
    cfg.dataloader.pin_memory
    and torch.cuda.is_available()
)

# ============================================================
# Train DataLoader
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    collate_fn=detection_collate_fn,
    pin_memory=pin_memory,
)

# ============================================================
# Validation DataLoader
# ============================================================

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=detection_collate_fn,
    pin_memory=pin_memory,
)

# ============================================================
# Test DataLoader
# ============================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=detection_collate_fn,
    pin_memory=pin_memory,
)

# ============================================================
# DataLoader 확인
# ============================================================

print("Train batch 수:", len(train_loader))
print("Validation batch 수:", len(valid_loader))
print("Test batch 수:", len(test_loader))

print("Batch size:", batch_size)
print("num_workers:", num_workers)
print("pin_memory:", pin_memory)

Train batch 수: 44
Validation batch 수: 4
Test batch 수: 7
Batch size: 4
num_workers: 0
pin_memory: False


In [24]:
# ============================================================
# DataLoader 첫 batch 동작 확인
# ============================================================

# detection_collate_fn이
# images / targets / metadata를 각각 list 형태로 반환하는지 확인합니다.
#
# 현재는 Faster R-CNN용 baseline transform이 연결되어 있으므로
# images 내부에는 torch.Tensor가 들어있는 것이 정상입니다.
#
# 최종 이미지 형식:
# [C, H, W]
# float32
# 값 범위 [0, 1]
# ============================================================

train_images, train_targets, train_metadata = next(
    iter(train_loader)
)

# ============================================================
# Batch 전체 구조 확인
# ============================================================

print(
    "images type:",
    type(train_images),
)

print(
    "targets type:",
    type(train_targets),
)

print(
    "metadata type:",
    type(train_metadata),
)

print(
    "\n배치 이미지 수:",
    len(train_images),
)

print(
    "배치 target 수:",
    len(train_targets),
)

print(
    "배치 metadata 수:",
    len(train_metadata),
)

# ============================================================
# 첫 번째 이미지 확인
# ============================================================

print(
    "\n===== 첫 번째 이미지 ====="
)

print(
    "Image type:",
    type(train_images[0]),
)

print(
    "Image shape:",
    train_images[0].shape,
)

print(
    "Image dtype:",
    train_images[0].dtype,
)

print(
    "Image min/max:",
    float(train_images[0].min()),
    float(train_images[0].max()),
)

# ============================================================
# 첫 번째 Target 확인
# ============================================================

print(
    "\n===== 첫 번째 Target ====="
)

print(
    "Target keys:",
    train_targets[0].keys(),
)

print(
    "Boxes shape:",
    train_targets[0]["boxes"].shape,
)

print(
    "Boxes:",
    train_targets[0]["boxes"],
)

print(
    "Labels:",
    train_targets[0]["labels"].tolist(),
)

# ============================================================
# 첫 번째 Metadata 확인
# ============================================================

print(
    "\n===== 첫 번째 Metadata ====="
)

print(
    "File name:",
    train_metadata[0]["file_name"],
)

print(
    "Combination key:",
    train_metadata[0]["combination_key"],
)

print(
    "Pill IDs:",
    train_metadata[0]["pill_ids"],
)

# ============================================================
# Faster R-CNN 입력 조건 검증
# ============================================================

assert torch.is_tensor(train_images[0])
assert train_images[0].dtype == torch.float32
assert train_images[0].ndim == 3
assert train_images[0].shape[0] == 3
assert tuple(train_images[0].shape[-2:]) == (640, 640)
assert train_images[0].min() >= 0.0
assert train_images[0].max() <= 1.0

print(
    "\nFaster R-CNN baseline transform 적용 확인 완료"
)

images type: <class 'list'>
targets type: <class 'list'>
metadata type: <class 'list'>

배치 이미지 수: 4
배치 target 수: 4
배치 metadata 수: 4

===== 첫 번째 이미지 =====
Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 640, 640])
Image dtype: torch.float32
Image min/max: 0.0 1.0

===== 첫 번째 Target =====
Target keys: dict_keys(['boxes', 'labels', 'image_id', 'area', 'iscrowd', 'annotation_id', 'ignore'])
Boxes shape: torch.Size([4, 4])
Boxes: tensor([[378.0000,  41.5000, 498.5000, 279.5000],
        [124.5000,  47.5000, 307.5000, 273.0000],
        [371.5000, 371.0000, 496.5000, 576.0000],
        [148.5000, 422.0000, 276.0000, 549.5000]])
Labels: [2, 22, 32, 35]

===== 첫 번째 Metadata =====
File name: K-002483-019552-022362-025438_0_2_0_2_90_000_200.png
Combination key: K-002483-019552-022362-025438
Pill IDs: ['002483', '019552', '022362', '025438']

Faster R-CNN baseline transform 적용 확인 완료


In [26]:
# ============================================================
# Random Seed 고정
# ============================================================
# 실험 재현성을 높이기 위해
# Python / NumPy / PyTorch의 random seed를 동일하게 고정합니다.
#
# seed 값은 config.yaml의
#
# cfg.project.seed
#
# 를 사용합니다.
#
# 현재 config.yaml에서는 seed=42로 설정되어 있습니다.
# 
# 추후 리팩토링 시 위치 상단으로 변경
# ============================================================

def set_seed(seed: int) -> None:

    # Python 기본 random
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch CPU
    torch.manual_seed(seed)

    # CUDA GPU 사용 시
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


# config.yaml의 seed 적용
set_seed(
    cfg.project.seed
)


print(
    "Random seed:",
    cfg.project.seed,
)

Random seed: 42


In [28]:
# ============================================================
# 학습 Device 설정
# ============================================================
# 사용 가능한 장치를 아래 우선순위로 선택합니다.
#
# 1. CUDA
#    → NVIDIA GPU 환경
#
# 2. MPS
#    → Apple Silicon(M1/M2/M3 등) Mac 환경
#
# 3. CPU
#
# 현재 로컬 Mac에서는 MPS가 선택될 가능성이 높습니다.
# ============================================================

if torch.cuda.is_available():

    device = torch.device("cuda")

elif torch.backends.mps.is_available():

    device = torch.device("mps")

else:

    device = torch.device("cpu")


# ============================================================
# 현재 선택된 Device 확인
# ============================================================

print(
    "Device:",
    device,
)


if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

elif device.type == "mps":

    print(
        "Apple Silicon MPS 사용 가능"
    )

else:

    print(
        "CPU 사용"
    )

Device: mps
Apple Silicon MPS 사용 가능


In [29]:
# ============================================================
# Weights & Biases 초기화 함수
# ============================================================
# W&B Run을 실제 학습 시작 시점에 생성하기 위한 함수입니다.
#
# 이 셀에서는 함수만 정의하며,
# wandb.login()이나 wandb.init()을 바로 실행하지 않습니다.
#
# 실제 Training Loop를 실행할 때 init_wandb(cfg)를 호출합니다.
#
# 기록할 주요 항목:
#
# - train loss
# - validation mAP
# - learning rate
# - epoch
# - Faster R-CNN 개별 loss
# - config.yaml 전체 설정
#
# 각 팀원은 자신의 W&B API Key로 로그인하고,
# 동일한 team entity / project를 사용하면
# 하나의 프로젝트에서 실험 결과를 함께 확인할 수 있습니다.
# ============================================================

import wandb


def init_wandb(cfg):

    # ========================================================
    # 1. W&B 사용 여부 확인
    # ========================================================
    # config.yaml에서 wandb.enabled=false인 경우
    # W&B를 사용하지 않고 None을 반환합니다.
    # ========================================================

    if not cfg.wandb.enabled:
        return None


    # ========================================================
    # 2. W&B 로그인
    # ========================================================
    # 최초 실행 시 API Key 입력을 요구할 수 있습니다.
    #
    # 이미 로그인된 환경에서는 저장된 인증 정보를 사용합니다.
    # ========================================================

    wandb.login()


    # ========================================================
    # 3. W&B Run 생성
    # ========================================================
    # project
    #   → W&B 프로젝트 이름
    #
    # entity
    #   → W&B 팀 entity 또는 개인 entity
    #
    # run_name
    #   → 현재 실험 이름
    #
    # tags
    #   → 실험 분류용 태그
    #
    # config
    #   → 현재 config.yaml 전체 설정을 W&B에 기록
    # ========================================================

    run = wandb.init(
        project=cfg.wandb.project,
        entity=cfg.wandb.entity,
        name=cfg.wandb.run_name,
        tags=list(cfg.wandb.tags),

        config=OmegaConf.to_container(
            cfg,
            resolve=True,
        ),
    )


    print(
        "W&B Run:",
        run.name,
    )


    return run

In [30]:
# ============================================================
# Faster R-CNN 모델 생성
# ============================================================
# torchvision의 Faster R-CNN + ResNet50 + FPN 모델을 사용합니다.
#
# pretrained=True이면 COCO pretrained weight를 불러옵니다.
#
# 기존 COCO classifier는 현재 알약 클래스 수와 맞지 않기 때문에
# 마지막 FastRCNNPredictor를 현재 Dataset 클래스 수에 맞게 교체합니다.
#
# Faster R-CNN에서는
#
# 0       : background
# 1 ~ N   : 실제 객체 클래스
#
# 구조를 사용하므로,
# Dataset의 알약 클래스 수 + 1을 최종 출력 클래스 수로 사용합니다.
# ============================================================

from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)

from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor,
)


def build_model(
    num_pill_classes: int,
    cfg,
):
    # ========================================================
    # 1. Pretrained weight 설정
    # ========================================================

    if cfg.model.pretrained:
        weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    else:
        weights = None


    # ========================================================
    # 2. Faster R-CNN 모델 생성
    # ========================================================

    model = fasterrcnn_resnet50_fpn(
        weights=weights,
        trainable_backbone_layers=(
            cfg.model.trainable_backbone_layers
        ),
        min_size=cfg.model.min_size,
        max_size=cfg.model.max_size,
    )


    # ========================================================
    # 3. 기존 COCO classifier 입력 feature 수 확인
    # ========================================================

    in_features = (
        model
        .roi_heads
        .box_predictor
        .cls_score
        .in_features
    )


    # ========================================================
    # 4. 현재 알약 클래스 수에 맞게 classifier 교체
    # ========================================================
    # Dataset 클래스 수가 56개라면
    #
    # background 1개
    # pill class 56개
    #
    # 총 57개 출력 클래스가 필요합니다.
    # ========================================================

    num_classes = (
        num_pill_classes + 1
    )


    model.roi_heads.box_predictor = (
        FastRCNNPredictor(
            in_features,
            num_classes,
        )
    )


    return model

In [31]:
# ============================================================
# 모델 생성 및 Device 이동
# ============================================================

model = build_model(
    num_pill_classes=dataset.num_classes,
    cfg=cfg,
)

model = model.to(device)


print(
    "알약 클래스 수:",
    dataset.num_classes,
)

print(
    "모델 출력 클래스 수:",
    dataset.num_classes + 1,
)

print(
    "Device:",
    device,
)

알약 클래스 수: 56
모델 출력 클래스 수: 57
Device: mps


In [32]:
# ============================================================
# Optimizer / Learning Rate Scheduler 설정
# ============================================================
# 학습 관련 하이퍼파라미터는 config.yaml 값을 사용합니다.
#
# optimizer
#   → sgd 또는 adamw
#
# learning_rate
# momentum
# weight_decay
#
# scheduler
#   → 현재는 StepLR 기준
#
# 이렇게 구성하면 이후 실험에서 config.yaml만 수정해서
# optimizer / learning rate / scheduler를 변경할 수 있습니다.
# ============================================================


# ============================================================
# 1. 학습 대상 Parameter 추출
# ============================================================
# requires_grad=True인 parameter만 optimizer에 전달합니다.
# ============================================================

trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]


# ============================================================
# 2. Optimizer 생성
# ============================================================

optimizer_name = (
    cfg.train.optimizer.lower()
)


if optimizer_name == "sgd":

    optimizer = torch.optim.SGD(
        trainable_parameters,
        lr=cfg.train.learning_rate,
        momentum=cfg.train.momentum,
        weight_decay=cfg.train.weight_decay,
    )


elif optimizer_name == "adamw":

    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=cfg.train.learning_rate,
        weight_decay=cfg.train.weight_decay,
    )


else:

    raise ValueError(
        f"지원하지 않는 optimizer입니다: "
        f"{cfg.train.optimizer}"
    )


# ============================================================
# 3. Scheduler 생성
# ============================================================

scheduler_type = (
    cfg.train.scheduler.type.lower()
)


if scheduler_type == "step":

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=cfg.train.scheduler.step_size,
        gamma=cfg.train.scheduler.gamma,
    )


elif scheduler_type == "none":

    scheduler = None


else:

    raise ValueError(
        f"지원하지 않는 scheduler입니다: "
        f"{cfg.train.scheduler.type}"
    )


# ============================================================
# 4. 현재 설정 확인
# ============================================================

print(
    "Optimizer:",
    optimizer_name,
)

print(
    "Learning rate:",
    optimizer.param_groups[0]["lr"],
)

print(
    "Weight decay:",
    cfg.train.weight_decay,
)

if optimizer_name == "sgd":
    print(
        "Momentum:",
        cfg.train.momentum,
    )

print(
    "Scheduler:",
    scheduler_type,
)

if scheduler is not None:

    print(
        "Step size:",
        cfg.train.scheduler.step_size,
    )

    print(
        "Gamma:",
        cfg.train.scheduler.gamma,
    )

Optimizer: sgd
Learning rate: 0.005
Weight decay: 0.0005
Momentum: 0.9
Scheduler: step
Step size: 5
Gamma: 0.1


In [34]:
# ============================================================
# Checkpoint 저장 경로 및 저장 함수
# ============================================================
# 모델 학습 중 아래 두 가지 checkpoint를 저장할 예정입니다.
#
# 1. faster_rcnn_last.pth
#    → 매 epoch 종료 후 마지막 학습 상태를 저장
#    → 학습이 중단되었을 때 이어서 학습하는 용도로 사용 가능
#
# 2. faster_rcnn_best.pth
#    → Validation 성능이 가장 좋은 모델을 저장
#    → 최종 추론 및 Kaggle 제출에 사용할 모델
#
# 저장 경로는 config.yaml의
#
# cfg.paths.checkpoint_dir
#
# 값을 사용합니다.
# ============================================================


# ============================================================
# 1. Checkpoint 디렉터리 생성
# ============================================================

checkpoint_dir = Path(
    cfg.paths.checkpoint_dir
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Checkpoint directory:",
    checkpoint_dir.resolve(),
)


# ============================================================
# 2. Checkpoint 저장 함수
# ============================================================
# 모델 weight뿐 아니라 학습을 이어서 진행할 수 있도록
#
# - epoch
# - model state
# - optimizer state
# - scheduler state
# - validation metric
#
# 을 함께 저장합니다.
# ============================================================

def save_checkpoint(
    model,
    optimizer,
    scheduler,
    epoch: int,
    val_map: float,
    checkpoint_path: Path,
) -> None:

    checkpoint = {
        "epoch": epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "val_map":
            val_map,
    }


    # Scheduler를 사용하는 경우에만 상태 저장
    if scheduler is not None:

        checkpoint[
            "scheduler_state_dict"
        ] = scheduler.state_dict()


    torch.save(
        checkpoint,
        checkpoint_path,
    )

Checkpoint directory: /Users/apple/dio_folder/python/codeit_cv_project/pill-object-detection/outputs/checkpoints


In [35]:
# ============================================================
# Faster R-CNN 학습 입력 형식 최종 확인
# ============================================================

# 팀원의 pill_transforms.py 기반 baseline transform이
# Dataset → DataLoader까지 정상적으로 적용되었는지 확인합니다.
#
# Faster R-CNN 입력 이미지 조건:
#
# - torch.Tensor
# - shape = [C, H, W]
# - C = 3
# - 640 x 640
# - dtype = torch.float32
# - 값 범위 = [0, 1]
# ============================================================

sample_images, sample_targets, sample_metadata = next(
    iter(train_loader)
)

sample_image = sample_images[0]
sample_target = sample_targets[0]

# ============================================================
# Image 확인
# ============================================================

print(
    "Image type:",
    type(sample_image),
)

print(
    "Image shape:",
    sample_image.shape,
)

print(
    "Image dtype:",
    sample_image.dtype,
)

print(
    "Image min/max:",
    float(sample_image.min()),
    float(sample_image.max()),
)

# ============================================================
# Target 확인
# ============================================================

print(
    "Boxes shape:",
    sample_target["boxes"].shape,
)

print(
    "Labels shape:",
    sample_target["labels"].shape,
)

# ============================================================
# 입력 조건 검증
# ============================================================

assert torch.is_tensor(sample_image), (
    "이미지가 torch.Tensor가 아닙니다."
)

assert sample_image.ndim == 3, (
    "이미지는 [C, H, W] 형태여야 합니다."
)

assert sample_image.shape[0] == 3, (
    "이미지 channel은 3이어야 합니다."
)

assert tuple(sample_image.shape[-2:]) == (640, 640), (
    f"이미지 크기가 640x640이 아닙니다: "
    f"{tuple(sample_image.shape[-2:])}"
)

assert sample_image.dtype == torch.float32, (
    "이미지 dtype은 torch.float32여야 합니다."
)

assert sample_image.min() >= 0.0
assert sample_image.max() <= 1.0

assert sample_target["boxes"].ndim == 2
assert sample_target["boxes"].shape[1] == 4

assert (
    len(sample_target["boxes"])
    == len(sample_target["labels"])
), (
    "bbox와 label 개수가 일치하지 않습니다."
)

print(
    "\nFaster R-CNN 학습 입력 형식 검증 완료"
)

Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 640, 640])
Image dtype: torch.float32
Image min/max: 0.0 1.0
Boxes shape: torch.Size([3, 4])
Labels shape: torch.Size([3])

Faster R-CNN 학습 입력 형식 검증 완료


In [36]:
# ============================================================
# Faster R-CNN 1 Epoch 학습 함수
# ============================================================
# 한 epoch 동안 train_loader의 모든 batch를 순회하면서
# Faster R-CNN의 loss를 계산하고 역전파를 수행합니다.
#
# Faster R-CNN은 model.train() 상태에서
#
# model(images, targets)
#
# 를 호출하면 prediction이 아니라 아래 loss dictionary를 반환합니다.
#
# loss_classifier
# loss_box_reg
# loss_objectness
# loss_rpn_box_reg
#
# 각 loss를 모두 더한 값을 기준으로 backward()를 수행합니다.
# ============================================================


def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
):

    # ========================================================
    # 1. 모델을 학습 모드로 설정
    # ========================================================

    model.train()


    # ========================================================
    # 2. Epoch 전체 loss 누적 변수
    # ========================================================

    total_loss_sum = 0.0


    # Faster R-CNN의 각 loss 항목도 따로 기록합니다.
    component_loss_sum = {
        "loss_classifier": 0.0,
        "loss_box_reg": 0.0,
        "loss_objectness": 0.0,
        "loss_rpn_box_reg": 0.0,
    }


    # ========================================================
    # 3. DataLoader batch 반복
    # ========================================================

    for (
        images,
        targets,
        _,
    ) in loader:


        # ====================================================
        # 4. 이미지와 target을 학습 device로 이동
        # ====================================================
        # Faster R-CNN은
        #
        # images  → Tensor list
        # targets → dictionary list
        #
        # 구조를 사용합니다.
        # ====================================================

        images = [
            image.to(device)
            for image in images
        ]


        targets = [
            {
                key: value.to(device)
                for key, value in target.items()
            }
            for target in targets
        ]


        # ====================================================
        # 5. 이전 batch gradient 초기화
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )


        # ====================================================
        # 6. Forward
        # ====================================================
        # Faster R-CNN train mode에서는
        # loss dictionary를 반환합니다.
        # ====================================================

        loss_dict = model(
            images,
            targets,
        )


        # ====================================================
        # 7. 전체 Loss 계산
        # ====================================================

        total_loss = sum(
            loss
            for loss in loss_dict.values()
        )


        # ====================================================
        # 8. Backpropagation
        # ====================================================

        total_loss.backward()


        # ====================================================
        # 9. Parameter 업데이트
        # ====================================================

        optimizer.step()


        # ====================================================
        # 10. Loss 누적
        # ====================================================

        total_loss_sum += (
            total_loss.item()
        )


        for (
            loss_name,
            loss_value,
        ) in loss_dict.items():

            component_loss_sum[
                loss_name
            ] += loss_value.item()


    # ========================================================
    # 11. Epoch 평균 Loss 계산
    # ========================================================

    num_batches = len(
        loader
    )


    avg_total_loss = (
        total_loss_sum
        / num_batches
    )


    avg_component_losses = {
        loss_name:
            loss_sum / num_batches

        for (
            loss_name,
            loss_sum,
        ) in component_loss_sum.items()
    }


    # ========================================================
    # 12. 결과 반환
    # ========================================================

    return (
        avg_total_loss,
        avg_component_losses,
    )

In [37]:
# ============================================================
# Validation mAP 계산 함수
# ============================================================
# Faster R-CNN의 Validation 성능을 mAP 기준으로 평가합니다.
#
# Object Detection에서는 단순 accuracy보다
# Bounding Box IoU와 클래스 예측을 함께 반영하는
# mAP(mean Average Precision)를 주로 사용합니다.
#
# 평가 흐름:
#
# 1. model.eval()로 추론 모드 설정
# 2. Validation 이미지에 대해 prediction 생성
# 3. prediction과 ground truth를 CPU로 이동
# 4. MeanAveragePrecision으로 mAP 계산
#
# 실제 competition 평가 기준이 확정되면
# IoU threshold는 해당 기준에 맞게 수정할 수 있습니다.
# ============================================================

from torchmetrics.detection.mean_ap import MeanAveragePrecision


@torch.no_grad()
def evaluate_map(
    model,
    loader,
    device,
):

    # ========================================================
    # 1. 모델을 평가 모드로 설정
    # ========================================================

    model.eval()


    # ========================================================
    # 2. mAP Metric 생성
    # ========================================================
    # 현재는 COCO 기본 평가 방식인
    # IoU 0.50 ~ 0.95 기준 mAP를 사용합니다.
    #
    # 추후 Kaggle 평가 기준이 별도로 정해져 있다면
    # iou_thresholds를 직접 지정할 수 있습니다.
    # ========================================================

    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[
            0.75,
            0.80,
            0.85,
            0.90,
            0.95,
        ],
    )


    # ========================================================
    # 3. Validation DataLoader 반복
    # ========================================================

    for (
        images,
        targets,
        _,
    ) in loader:


        # ====================================================
        # 4. 이미지를 Device로 이동
        # ====================================================
        # model.eval() 상태에서는 target을 모델에 전달하지 않고
        # images만 입력하여 prediction을 얻습니다.
        # ====================================================

        images = [
            image.to(device)
            for image in images
        ]


        # ====================================================
        # 5. Faster R-CNN Prediction
        # ====================================================
        # 각 이미지별로 다음 값을 반환합니다.
        #
        # boxes
        # labels
        # scores
        # ====================================================

        predictions = model(
            images
        )


        # ====================================================
        # 6. Prediction을 CPU로 이동
        # ====================================================
        # torchmetrics에서 평가할 수 있도록
        # GPU/MPS Tensor를 CPU Tensor로 변환합니다.
        # ====================================================

        predictions_cpu = []

        for prediction in predictions:

            predictions_cpu.append(
                {
                    "boxes":
                        prediction["boxes"]
                        .detach()
                        .cpu(),

                    "scores":
                        prediction["scores"]
                        .detach()
                        .cpu(),

                    "labels":
                        prediction["labels"]
                        .detach()
                        .cpu(),
                }
            )


        # ====================================================
        # 7. Ground Truth를 CPU 형식으로 구성
        # ====================================================
        # mAP 계산에는 boxes와 labels가 필요합니다.
        # ====================================================

        targets_cpu = []

        for target in targets:

            targets_cpu.append(
                {
                    "boxes":
                        target["boxes"]
                        .detach()
                        .cpu(),

                    "labels":
                        target["labels"]
                        .detach()
                        .cpu(),
                }
            )


        # ====================================================
        # 8. 현재 Batch 결과 누적
        # ====================================================

        metric.update(
            predictions_cpu,
            targets_cpu,
        )


    # ========================================================
    # 9. 전체 Validation Dataset mAP 계산
    # ========================================================

    results = metric.compute()


    # ========================================================
    # 10. 결과 반환
    # ========================================================
    # 주요 결과:
    #
    # results["map"]
    #   → IoU 0.50:0.95 평균 mAP
    #
    # results["map_50"]
    #   → IoU 0.50 기준 mAP
    #
    # results["map_75"]
    #   → IoU 0.75 기준 mAP
    # ========================================================

    return results

In [38]:
# ============================================================
# Faster R-CNN 전체 학습 함수
# ============================================================

# 전체 학습 과정을 하나의 함수에서 관리합니다.
#
# Validation 기준:
# → competition 평가 기준과 동일한 mAP@[0.75:0.95]
#
# best checkpoint 역시 해당 mAP를 기준으로 저장합니다.
# ============================================================

def train_model(
    model,
    train_loader,
    valid_loader,
    optimizer,
    scheduler,
    device,
    checkpoint_dir,
    cfg,
):

    # ========================================================
    # 1. W&B Run 초기화
    # ========================================================

    run = init_wandb(
        cfg
    )

    # ========================================================
    # 2. Best Validation mAP 초기값
    # ========================================================

    best_map = -1.0

    try:

        # ====================================================
        # 3. Epoch 반복
        # ====================================================

        for epoch in range(
            1,
            cfg.train.epochs + 1,
        ):

            print(
                "\n"
                "========================================"
            )

            print(
                f"Epoch "
                f"{epoch}/{cfg.train.epochs}"
            )

            print(
                "========================================"
            )

            # =================================================
            # 4. Train
            # =================================================

            (
                train_loss,
                train_loss_dict,
            ) = train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                device=device,
            )

            # =================================================
            # 5. Validation
            # =================================================

            val_metrics = evaluate_map(
                model=model,
                loader=valid_loader,
                device=device,
            )

            # Competition 기준
            # mAP@[0.75:0.95]
            val_map = float(
                val_metrics["map"]
            )

            # IoU 0.75 단일 mAP
            val_map75 = float(
                val_metrics["map_75"]
            )

            # =================================================
            # 6. 현재 Learning Rate
            # =================================================

            current_lr = (
                optimizer
                .param_groups[0]
                ["lr"]
            )

            # =================================================
            # 7. 현재 Epoch 결과 출력
            # =================================================

            print(
                f"Train Loss        : "
                f"{train_loss:.4f}"
            )

            print(
                f"Val mAP@0.75:0.95 : "
                f"{val_map:.4f}"
            )

            print(
                f"Val mAP@0.75      : "
                f"{val_map75:.4f}"
            )

            print(
                f"Learning Rate     : "
                f"{current_lr:.8f}"
            )

            # =================================================
            # 8. W&B Logging
            # =================================================

            if run is not None:

                log_data = {
                    "epoch":
                        epoch,

                    "train/loss":
                        train_loss,

                    "val/map_75_95":
                        val_map,

                    "val/map_75":
                        val_map75,

                    "learning_rate":
                        current_lr,
                }

                # Faster R-CNN 개별 loss 기록
                for (
                    loss_name,
                    loss_value,
                ) in train_loss_dict.items():

                    log_data[
                        f"train/{loss_name}"
                    ] = loss_value

                wandb.log(
                    log_data
                )

            # =================================================
            # 9. Last Checkpoint 저장
            # =================================================

            last_checkpoint_path = (
                checkpoint_dir
                / "faster_rcnn_last.pth"
            )

            save_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                val_map=val_map,
                checkpoint_path=(
                    last_checkpoint_path
                ),
            )

            # =================================================
            # 10. Best Checkpoint 저장
            # =================================================
            # Competition 기준인 mAP@[0.75:0.95]를 기준으로
            # best model을 선택합니다.
            # =================================================

            if (
                cfg.train.save_best
                and val_map > best_map
            ):

                best_map = val_map

                best_checkpoint_path = (
                    checkpoint_dir
                    / "faster_rcnn_best.pth"
                )

                save_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch,
                    val_map=val_map,
                    checkpoint_path=(
                        best_checkpoint_path
                    ),
                )

                print(
                    f"Best model 저장 완료 "
                    f"(mAP@0.75:0.95="
                    f"{best_map:.4f})"
                )

            # =================================================
            # 11. Learning Rate Scheduler 업데이트
            # =================================================

            if scheduler is not None:

                scheduler.step()

    # ========================================================
    # 12. W&B Run 종료
    # ========================================================

    finally:

        if run is not None:

            wandb.finish()

    # ========================================================
    # 13. 최고 Validation mAP 반환
    # ========================================================

    return best_map

In [ ]:
# ============================================================
# Faster R-CNN 학습 실행
# ============================================================
# 재웅님이 작성한 pill_transforms.py 기반 baseline transform 연결 완료
# DataLoader 이미지 Tensor 검증 완료
#
# Validation 기준:
# mAP@[0.75:0.95]
# ============================================================

faster_rcnn_best_map = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    checkpoint_dir=checkpoint_dir,
    cfg=cfg,
)


print(
    f"최종 Best Validation mAP@0.75:0.95: "
    f"{faster_rcnn_best_map:.4f}"
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/apple/.netrc.
wandb: Currently logged in as: dykim335 (diokim17) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B Run: faster-rcnn-baseline

Epoch 1/20
